# Renovación programada de tokens Arcade — CIREN

Actualiza las variables `var token` de los popups Arcade sin mostrar el token en salidas. El proceso recorre el JSON completo de Web Maps/Dashboards y utiliza Regex compatible con comillas simples, dobles y saltos de línea.

In [ ]:
from pathlib import Path
from IPython.display import display
from Lib.esrilogs import Logfile, capturaError
from configuracion_ciren import load_solution_config, connect_gis_from_config
from actualizar_tokens_arcade import refresh_item_tokens

In [ ]:
CONFIG_PATH = Path('configuracion_ciren.json')
config = load_solution_config(str(CONFIG_PATH))
ITEM_IDS = config['arcade_tokens']['item_ids']
EXPECTED_MATCHES = config['arcade_tokens']['expected_matches']
EXECUTE_CHANGES = config['arcade_tokens']['notebook_execute_changes']
log_config = config['logs']
logs = Logfile(
    'ActualizarTokensArcadeNotebook',
    log_path=Path(log_config['path']),
    max_age_days=log_config.get('max_age_days', 30),
    rotate_mode=log_config.get('rotate_mode', 'archive'),
)
logs.start_script('Inicio notebook de tokens Arcade')

In [ ]:
gis = connect_gis_from_config(config)
logs.info('Conexion autenticada; el token no se mostrara')

## Simulación
Localiza las expresiones y muestra sus rutas JSON, pero no modifica ítems.

In [ ]:
simulation = refresh_item_tokens(gis, ITEM_IDS, dry_run=True, logs=logs)
display(simulation)
logs.info(f'Reporte de simulacion: {simulation}')
assert sum(item['matches'] for item in simulation) == EXPECTED_MATCHES, 'Cantidad de tokens distinta de la configuración.'

## Actualización
Cambie `EXECUTE_CHANGES` a `True` para renovar los tokens. Para una tarea programada puede invocarse directamente el módulo desde un script, manteniendo las credenciales fuera del código.

In [ ]:
if not EXECUTE_CHANGES:
    print('Actualización bloqueada: simulación solamente.')
else:
    result = refresh_item_tokens(gis, ITEM_IDS, dry_run=False, logs=logs)
    display(result)
    logs.end(f'Tokens actualizados: {result}')
logs.close('Notebook de tokens finalizado')